# Convert JSON

## indspeech_news_tts

In [1]:
import csv
import json
import os

# path folder wav
wav_dir = "/home/nafis/code/text-to-speech/data/data/indspeech_news_tts/wavs"
# input csv
csv_file = "/home/nafis/code/text-to-speech/data/data/indspeech_news_tts/dia_dataset.csv"
# output json
json_file = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/indspeech_news_tts.json"

data = []

with open(csv_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")  # karena pakai | di contohmu
    for row in reader:
        if len(row) < 2:
            continue  # skip baris yang ga valid
        filename = row[0].strip()
        text = row[1].strip()
        
        audio_path = os.path.join(wav_dir, filename)
        if not os.path.exists(audio_path):
            print(f"Warning: file {audio_path} tidak ditemukan.")
            continue

        # Ambil speaker ID dari nama file
        # Contoh: SPK00_F001.wav → SPK00
        speaker = "indspeech_news_tts_0"

        data.append({
            "audio_path": audio_path,
            "text": text,
            "speaker": speaker
        })

# simpan jadi JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"JSON dataset tersimpan di {json_file}, total {len(data)} item")

JSON dataset tersimpan di /home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/indspeech_news_tts.json, total 2012 item


In [11]:
import torchaudio
import os

def get_total_audio_duration_and_count(folder_path, extensions={".wav", ".mp3", ".flac"}):
    total_seconds = 0.0
    file_count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                path = os.path.join(root, file)
                try:
                    waveform, sample_rate = torchaudio.load(path)
                    duration = waveform.shape[1] / sample_rate
                    total_seconds += duration
                    file_count += 1
                except Exception as e:
                    print(f"Failed to load {file}: {e}")
    total_hours = total_seconds / 3600
    return total_hours, file_count

# Ganti dengan path ke folder kamu
folder = "/home/nafis/code/text-to-speech/data/data/indspeech_news_tts/wavs"
hours, count = get_total_audio_duration_and_count(folder)
print(f"Total audio duration: {hours:.2f} hours")
print(f"Total number of audio files: {count}")


Total audio duration: 1.93 hours
Total number of audio files: 2012


In [ ]:
CUDA_VISIBLE_DEVICES=2 python convert_dac.py --path "/home/nafis/code/text-to-speech/data/data/indspeech_news_tts/wavs/*.wav"

## sindodusc

In [1]:
from datasets import load_dataset

path = "/home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/asr_sindodusc"
dataset = load_dataset(path)
dataset

/home/nafis/.pyenv/versions/3.10.0/envs/dia/lib/python3.10/site-packages/datasets/load.py:929: FutureWarning: The repository for asr_sindodusc contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/asr_sindodusc/asr_sindodusc.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'channel', 'uttrans_id', 'speaker_id', 'transcription', 'path', 'audio', 'speaker_gender', 'speaker_age', 'speaker_region', 'speaker_device'],
        num_rows: 3296
    })
})

In [5]:
import os
from datasets import load_dataset
from pathlib import Path
import soundfile as sf

dataset_path = "/home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/asr_sindodusc"
dataset = load_dataset(dataset_path)

audio_folder = "/home/nafis/code/text-to-speech/data/data/sindodusc/wavs"
os.makedirs(audio_folder, exist_ok=True)

txt_file = "/home/nafis/code/text-to-speech/data/data/sindodusc/metadata.txt"

with open(txt_file, "w", encoding="utf-8") as f:
    for item in dataset["train"]:
        # simpan audio
        audio_array = item["audio"]["array"]
        sampling_rate = item["audio"]["sampling_rate"]
        audio_path = os.path.join(audio_folder, item["uttrans_id"])
        sf.write(audio_path, audio_array, sampling_rate)

        # bersihkan transkripsi
        transcription = item["transcription"].replace("\t", " ").replace("\n", " ").strip()

        # tulis ke txt dengan delimiter |
        f.write(f"{item['uttrans_id']}|{transcription}\n")

In [2]:
import csv
import json
import os

# path folder wav
wav_dir = "/home/nafis/code/text-to-speech/data/data/sindodusc/wavs"
# input csv
csv_file = "/home/nafis/code/text-to-speech/data/data/sindodusc/metadata.txt"
# output json
json_file = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/sindodusc.json"

data = []

with open(csv_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")  # karena pakai | di contohmu
    for row in reader:
        if len(row) < 2:
            continue  # skip baris yang ga valid
        filename = row[0].strip()
        text = row[1].strip()
        
        audio_path = os.path.join(wav_dir, filename)
        if not os.path.exists(audio_path):
            print(f"Warning: file {audio_path} tidak ditemukan.")
            continue

        # Ambil speaker ID dari nama file
        speaker = filename.split("_")[0]
        sindodusc_filename = f"sindodusc_{speaker}"

        data.append({
            "audio_path": audio_path,
            "text": text,
            "speaker": sindodusc_filename
        })

# simpan jadi JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"JSON dataset tersimpan di {json_file}, total {len(data)} item")

JSON dataset tersimpan di /home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/sindodusc.json, total 3296 item


In [12]:
import torchaudio
import os

def get_total_audio_duration_and_count(folder_path, extensions={".wav", ".mp3", ".flac"}):
    total_seconds = 0.0
    file_count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                path = os.path.join(root, file)
                try:
                    waveform, sample_rate = torchaudio.load(path)
                    duration = waveform.shape[1] / sample_rate
                    total_seconds += duration
                    file_count += 1
                except Exception as e:
                    print(f"Failed to load {file}: {e}")
    total_hours = total_seconds / 3600
    return total_hours, file_count

# Ganti dengan path ke folder kamu
folder = "/home/nafis/code/text-to-speech/data/data/sindodusc/wavs"
hours, count = get_total_audio_duration_and_count(folder)
print(f"Total audio duration: {hours:.2f} hours")
print(f"Total number of audio files: {count}")


Total audio duration: 3.62 hours
Total number of audio files: 3296


In [ ]:
CUDA_VISIBLE_DEVICES=2 python convert_dac.py --path "/home/nafis/code/text-to-speech/data/data/sindodusc/wavs/*.wav"

## TITML-IDN

In [6]:
import os
import soundfile as sf
from datasets import load_dataset

dataset = load_dataset("atmatechai/TITML-IDN")

audio_folder = "/home/nafis/code/text-to-speech/data/data/titml-idn/wavs"
os.makedirs(audio_folder, exist_ok=True)

txt_file = "/home/nafis/code/text-to-speech/data/data/titml-idn/metadata.txt"

with open(txt_file, "w", encoding="utf-8") as f:
    for split in ["train", "test"]:
        for item in dataset[split]:
            # ambil nama file dari path
            filename = os.path.basename(item['audio']['path'])
            
            # speaker ID dari bagian awal filename (misal 07-246.wav → 07)
            speaker = filename.split("_")[0]
            sindodusc_filename = f"titml-idn_{speaker}"
            
            # path tujuan audio
            audio_path = os.path.join(audio_folder, filename)
            
            # simpan audio
            sf.write(audio_path, item['audio']['array'], item['audio']['sampling_rate'])
            
            # bersihkan transkripsi
            transcription = item['sentence'].replace("\t", " ").replace("\n", " ").strip()
            
            # tulis ke txt
            f.write(f"{filename}|{transcription}\n")


In [1]:
import csv
import json
import os

# path folder wav
wav_dir = "/home/nafis/code/text-to-speech/data/data/titml-idn/wavs"
# input csv
csv_file = "/home/nafis/code/text-to-speech/data/data/titml-idn/metadata.txt"
# output json
json_file = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/titml-idn.json"

data = []

with open(csv_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")  # karena pakai | di contohmu
    for row in reader:
        if len(row) < 2:
            continue  # skip baris yang ga valid
        filename = row[0].strip()
        text = row[1].strip()
        
        audio_path = os.path.join(wav_dir, filename)
        if not os.path.exists(audio_path):
            print(f"Warning: file {audio_path} tidak ditemukan.")
            continue

        # Ambil speaker ID dari nama file
        speaker = filename.split("_")[0]
        titml_idn_filename = f"titml-idn_{speaker}"

        data.append({
            "audio_path": audio_path,
            "text": text,
            "speaker": titml_idn_filename
        })

# simpan jadi JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"JSON dataset tersimpan di {json_file}, total {len(data)} item")

JSON dataset tersimpan di /home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/titml-idn.json, total 6679 item


In [10]:
import torchaudio
import os

def get_total_audio_duration_and_count(folder_path, extensions={".wav", ".mp3", ".flac"}):
    total_seconds = 0.0
    file_count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                path = os.path.join(root, file)
                try:
                    waveform, sample_rate = torchaudio.load(path)
                    duration = waveform.shape[1] / sample_rate
                    total_seconds += duration
                    file_count += 1
                except Exception as e:
                    print(f"Failed to load {file}: {e}")
    total_hours = total_seconds / 3600
    return total_hours, file_count

# Ganti dengan path ke folder kamu
folder = "/home/nafis/code/text-to-speech/data/data/titml-idn/wavs"
hours, count = get_total_audio_duration_and_count(folder)
print(f"Total audio duration: {hours:.2f} hours")
print(f"Total number of audio files: {count}")


Total audio duration: 14.52 hours
Total number of audio files: 6679


In [ ]:
CUDA_VISIBLE_DEVICES=2 python convert_dac.py --path "/home/nafis/code/text-to-speech/data/data/titml-idn/wavs/*.wav"

## MEDISCO

In [1]:
from datasets import load_dataset

path = "/home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/medisco"
dataset = load_dataset(path)
dataset

/home/nafis/.pyenv/versions/3.10.0/envs/dia/lib/python3.10/site-packages/datasets/load.py:929: FutureWarning: The repository for medisco contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/medisco/medisco.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'path', 'audio', 'text'],
        num_rows: 3960
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'path', 'audio', 'text'],
        num_rows: 720
    })
})

In [24]:
speaker_ids = dataset['train']["speaker_id"]
from collections import Counter
speaker_id_counts = Counter(speaker_ids)

for speaker_id, count in speaker_id_counts.items():
    print("speaker_id:", speaker_id, "Jumlah:", count)

speaker_id: female-2 Jumlah: 360
speaker_id: male-3 Jumlah: 360
speaker_id: male-5 Jumlah: 360
speaker_id: male-4 Jumlah: 360
speaker_id: male-2 Jumlah: 360
speaker_id: female-1 Jumlah: 360
speaker_id: female-4 Jumlah: 360
speaker_id: female-3 Jumlah: 360
speaker_id: male-1 Jumlah: 360
speaker_id: male-6 Jumlah: 360
speaker_id: female-5 Jumlah: 360


In [25]:
speaker_ids = dataset['test']["speaker_id"]
from collections import Counter
speaker_id_counts = Counter(speaker_ids)

for speaker_id, count in speaker_id_counts.items():
    print("speaker_id:", speaker_id, "Jumlah:", count)

speaker_id: male Jumlah: 360
speaker_id: female Jumlah: 360


In [8]:
import os
import soundfile as sf
from datasets import load_dataset

# load dataset
path = "/home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/medisco"
dataset = load_dataset(path)

# folder audio
audio_folder = "/home/nafis/code/text-to-speech/data/data/medisco/wavs"
os.makedirs(audio_folder, exist_ok=True)

# file txt untuk transkripsi
txt_file = "/home/nafis/code/text-to-speech/data/data/medisco/metadata.txt"

# mapping speaker test
test_speaker_map = {
    "male": "male-7",
    "female": "female-6"
}

with open(txt_file, "w", encoding="utf-8") as f:
    # loop train
    for item in dataset["train"]:
        speaker = item['speaker_id']  # ambil langsung speaker_id train
        number_id = item['id'].split("_")[-1]  # ambil nomor dari id
        filename = f"{speaker}_{number_id}.wav"
        
        audio_path = os.path.join(audio_folder, filename)
        sf.write(audio_path, item['audio']['array'], item['audio']['sampling_rate'])
        
        transcription = item['text'].replace("\t", " ").replace("\n", " ").strip()
        f.write(f"{filename}|{transcription}\n")
    
    # loop test
    for item in dataset["test"]:
        speaker = test_speaker_map.get(item['speaker_id'].lower(), item['speaker_id'])
        number_id = item['id'].split("_")[-1]
        filename = f"{speaker}_{number_id}.wav"
        
        audio_path = os.path.join(audio_folder, filename)
        sf.write(audio_path, item['audio']['array'], item['audio']['sampling_rate'])
        
        transcription = item['text'].replace("\t", " ").replace("\n", " ").strip()
        f.write(f"{filename}|{transcription}\n")

/home/nafis/.pyenv/versions/3.10.0/envs/dia/lib/python3.10/site-packages/datasets/load.py:929: FutureWarning: The repository for medisco contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/medisco/medisco.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [14]:
from IPython.display import Audio

audio_file = "/home/nafis/code/text-to-speech/data/data/medisco/wavs/female-6_66.wav"
Audio(audio_file)  # otomatis membaca file WAV


In [15]:
import csv
import json
import os

# path folder wav
wav_dir = "/home/nafis/code/text-to-speech/data/data/medisco/wavs"
# input csv
csv_file = "/home/nafis/code/text-to-speech/data/data/medisco/metadata.txt"
# output json
json_file = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/medisco.json"

data = []

with open(csv_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")  # karena pakai | di contohmu
    for row in reader:
        if len(row) < 2:
            continue  # skip baris yang ga valid
        filename = row[0].strip()
        text = row[1].strip()
        
        audio_path = os.path.join(wav_dir, filename)
        if not os.path.exists(audio_path):
            print(f"Warning: file {audio_path} tidak ditemukan.")
            continue

        # Ambil speaker ID dari nama file
        speaker = filename.split("_")[0]
        medisco_filename = f"medisco_{speaker}"

        data.append({
            "audio_path": audio_path,
            "text": text,
            "speaker": medisco_filename
        })

# simpan jadi JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"JSON dataset tersimpan di {json_file}, total {len(data)} item")

JSON dataset tersimpan di /home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/medisco.json, total 4680 item


In [9]:
import torchaudio
import os

def get_total_audio_duration_and_count(folder_path, extensions={".wav", ".mp3", ".flac"}):
    total_seconds = 0.0
    file_count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                path = os.path.join(root, file)
                try:
                    waveform, sample_rate = torchaudio.load(path)
                    duration = waveform.shape[1] / sample_rate
                    total_seconds += duration
                    file_count += 1
                except Exception as e:
                    print(f"Failed to load {file}: {e}")
    total_hours = total_seconds / 3600
    return total_hours, file_count

# Ganti dengan path ke folder kamu
folder = "/home/nafis/code/text-to-speech/data/data/medisco/wavs"
hours, count = get_total_audio_duration_and_count(folder)
print(f"Total audio duration: {hours:.2f} hours")
print(f"Total number of audio files: {count}")


Total audio duration: 10.03 hours
Total number of audio files: 4680


In [ ]:
CUDA_VISIBLE_DEVICES=2 nohup python convert_dac.py --path "/home/nafis/code/text-to-speech/data/data/medisco/wavs/*.wav" > convert_dac.out &

## indspeech_teldialog_lvcsr

In [1]:
from datasets import load_dataset

path = "/home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/indspeech_teldialog_lvcsr"
dataset = load_dataset(path)
dataset

/home/nafis/.pyenv/versions/3.10.0/envs/dia/lib/python3.10/site-packages/datasets/load.py:929: FutureWarning: The repository for indspeech_teldialog_lvcsr contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/indspeech_teldialog_lvcsr/indspeech_teldialog_lvcsr.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'path', 'audio', 'text'],
        num_rows: 36000
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'path', 'audio', 'text'],
        num_rows: 4000
    })
})

In [3]:
speaker_ids = dataset['train']["speaker_id"]
from collections import Counter
speaker_id_counts = Counter(speaker_ids)

for speaker_id, count in speaker_id_counts.items():
    print("speaker_id:", speaker_id, "Jumlah:", count)

speaker_id: Ind001_F_B Jumlah: 100
speaker_id: Ind002_M_S Jumlah: 100
speaker_id: Ind003_M_U Jumlah: 100
speaker_id: Ind004_F_S Jumlah: 100
speaker_id: Ind005_F_S Jumlah: 100
speaker_id: Ind006_M_J Jumlah: 100
speaker_id: Ind007_M_J Jumlah: 100
speaker_id: Ind008_F_U Jumlah: 100
speaker_id: Ind009_F_U Jumlah: 100
speaker_id: Ind010_F_S Jumlah: 100
speaker_id: Ind011_F_U Jumlah: 100
speaker_id: Ind012_M_J Jumlah: 100
speaker_id: Ind013_M_U Jumlah: 100
speaker_id: Ind014_M_U Jumlah: 100
speaker_id: Ind015_F_S Jumlah: 100
speaker_id: Ind016_M_J Jumlah: 100
speaker_id: Ind017_F_S Jumlah: 100
speaker_id: Ind018_F_S Jumlah: 100
speaker_id: Ind019_F_U Jumlah: 100
speaker_id: Ind020_F_J Jumlah: 100
speaker_id: Ind021_M_J Jumlah: 100
speaker_id: Ind022_M_U Jumlah: 100
speaker_id: Ind023_M_J Jumlah: 100
speaker_id: Ind024_F_S Jumlah: 100
speaker_id: Ind025_F_S Jumlah: 100
speaker_id: Ind026_F_S Jumlah: 100
speaker_id: Ind027_F_S Jumlah: 100
speaker_id: Ind028_F_J Jumlah: 100
speaker_id: Ind029_F

In [4]:
speaker_ids = dataset['test']["speaker_id"]
from collections import Counter
speaker_id_counts = Counter(speaker_ids)

for speaker_id, count in speaker_id_counts.items():
    print("speaker_id:", speaker_id, "Jumlah:", count)

speaker_id: Ind239_M_J Jumlah: 100
speaker_id: Ind250_M_J Jumlah: 100
speaker_id: Ind255_M_J Jumlah: 100
speaker_id: Ind276_M_J Jumlah: 100
speaker_id: Ind335_M_B Jumlah: 100
speaker_id: Ind338_M_U Jumlah: 100
speaker_id: Ind340_M_U Jumlah: 100
speaker_id: Ind341_M_S Jumlah: 100
speaker_id: Ind342_M_S Jumlah: 100
speaker_id: Ind343_M_B Jumlah: 100
speaker_id: Ind345_M_U Jumlah: 100
speaker_id: Ind346_M_B Jumlah: 100
speaker_id: Ind347_M_B Jumlah: 100
speaker_id: Ind350_F_S Jumlah: 100
speaker_id: Ind353_M_B Jumlah: 100
speaker_id: Ind354_F_S Jumlah: 100
speaker_id: Ind355_F_S Jumlah: 100
speaker_id: Ind361_M_B Jumlah: 100
speaker_id: Ind364_M_S Jumlah: 100
speaker_id: Ind366_M_S Jumlah: 100
speaker_id: Ind368_F_S Jumlah: 100
speaker_id: Ind369_M_U Jumlah: 100
speaker_id: Ind375_M_S Jumlah: 100
speaker_id: Ind376_M_U Jumlah: 100
speaker_id: Ind380_F_S Jumlah: 100
speaker_id: Ind381_F_S Jumlah: 100
speaker_id: Ind382_F_B Jumlah: 100
speaker_id: Ind383_F_B Jumlah: 100
speaker_id: Ind387_F

In [10]:
import os
import soundfile as sf
from datasets import load_dataset

path = "/home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/indspeech_teldialog_lvcsr"
dataset = load_dataset(path)

audio_folder = "/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs"
os.makedirs(audio_folder, exist_ok=True)

txt_file = "/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/metadata.txt"

with open(txt_file, "w", encoding="utf-8") as f:
    for split in ["train", "test"]:
        for item in dataset[split]:
            # filename pakai id + .wav
            filename = f"{item['id']}.wav"
            
            # path tujuan audio
            audio_path = os.path.join(audio_folder, filename)
            
            # simpan audio
            sf.write(audio_path, item['audio']['array'], item['audio']['sampling_rate'])
            
            # ambil transkripsi
            transcription = item['text'].replace("\t", " ").replace("\n", " ").strip()
            
            # tulis ke txt
            f.write(f"{filename}|{transcription}\n")


/home/nafis/.pyenv/versions/3.10.0/envs/dia/lib/python3.10/site-packages/datasets/load.py:929: FutureWarning: The repository for indspeech_teldialog_lvcsr contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/nafis/code/text-to-speech/data/seacrowd-datahub/seacrowd/sea_datasets/indspeech_teldialog_lvcsr/indspeech_teldialog_lvcsr.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [13]:
dataset["train"][9]

{'id': 'Ind001_F_B_C_appl_0091',
 'speaker_id': 'Ind001_F_B',
 'path': '/home/nafis/.cache/huggingface/datasets/downloads/extracted/b0ff8b6f9b5b69183aa50240e27ab9b5bfc10a25fb3470481306aa0ecc52e76c/Ind001/Ind001_F_B_C_appl_0091.wav',
 'audio': {'path': '/home/nafis/.cache/huggingface/datasets/downloads/extracted/b0ff8b6f9b5b69183aa50240e27ab9b5bfc10a25fb3470481306aa0ecc52e76c/Ind001/Ind001_F_B_C_appl_0091.wav',
  'array': array([ 3.96728516e-04,  1.22070312e-04,  3.05175781e-04, ...,
         -1.22070312e-04,  2.74658203e-04, -6.10351562e-05], shape=(48640,)),
  'sampling_rate': 16000},
 'text': 'Mohon info tentang KK.'}

In [14]:
import soundfile as sf
from IPython.display import Audio, display

audio_file="/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs/Ind001_F_B_C_appl_0091.wav"

audio_array, sr = sf.read(audio_file)
display(Audio(audio_array, rate=sr))


In [1]:
import csv
import json
import os

# path folder wav
wav_dir = "/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs"
# input csv
csv_file = "/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/metadata.txt"
# output json
json_file = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/indspeech_teldialog_lvcsr.json"

data = []

with open(csv_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")  # karena pakai | di contohmu
    for row in reader:
        if len(row) < 2:
            continue  # skip baris yang ga valid
        filename = row[0].strip()
        text = row[1].strip()
        
        audio_path = os.path.join(wav_dir, filename)
        if not os.path.exists(audio_path):
            print(f"Warning: file {audio_path} tidak ditemukan.")
            continue

        # Ambil speaker ID dari nama file
        parts = filename.split("_")
        speaker = "_".join(parts[0:4]) 
        teldialog_filename = f"teldialog_{speaker}"

        data.append({
            "audio_path": audio_path,
            "text": text,
            "speaker": teldialog_filename
        })

# simpan jadi JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"JSON dataset tersimpan di {json_file}, total {len(data)} item")

JSON dataset tersimpan di /home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/indspeech_teldialog_lvcsr.json, total 40000 item


In [8]:
import torchaudio
import os

def get_total_audio_duration_and_count(folder_path, extensions={".wav", ".mp3", ".flac"}):
    total_seconds = 0.0
    file_count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                path = os.path.join(root, file)
                try:
                    waveform, sample_rate = torchaudio.load(path)
                    duration = waveform.shape[1] / sample_rate
                    total_seconds += duration
                    file_count += 1
                except Exception as e:
                    print(f"Failed to load {file}: {e}")
    total_hours = total_seconds / 3600
    return total_hours, file_count

# Ganti dengan path ke folder kamu
folder = "/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs"
hours, count = get_total_audio_duration_and_count(folder)
print(f"Total audio duration: {hours:.2f} hours")
print(f"Total number of audio files: {count}")


Total audio duration: 36.15 hours
Total number of audio files: 40000


In [ ]:
CUDA_VISIBLE_DEVICES=2 nohup python convert_dac.py --path "/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs/*.wav" > convert_dac.out & 

## tito

In [2]:
import csv
import json
import os

# path folder wav
wav_dir = "/home/nafis/code/text-to-speech/data/data/tito/wavs"
# input csv
csv_file = "/home/nafis/code/text-to-speech/data/data/tito/tito_transcript.txt"
# output json
json_file = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/tito.json"

data = []

with open(csv_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")  # karena pakai | di contohmu
    for row in reader:
        if len(row) < 2:
            continue  # skip baris yang ga valid
        filename = row[0].strip()
        text = row[1].strip()
        
        audio_path = os.path.join(wav_dir, filename)
        if not os.path.exists(audio_path):
            print(f"Warning: file {audio_path} tidak ditemukan.")
            continue

        # Ambil speaker ID dari nama file
        tito_filename = f"tito"

        data.append({
            "audio_path": audio_path,
            "text": text,
            "speaker": tito_filename
        })

# simpan jadi JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"JSON dataset tersimpan di {json_file}, total {len(data)} item")

JSON dataset tersimpan di /home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/tito.json, total 3349 item


In [3]:
import torchaudio
import os

def get_total_audio_duration_and_count(folder_path, extensions={".wav", ".mp3", ".flac"}):
    total_seconds = 0.0
    file_count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                path = os.path.join(root, file)
                try:
                    waveform, sample_rate = torchaudio.load(path)
                    duration = waveform.shape[1] / sample_rate
                    total_seconds += duration
                    file_count += 1
                except Exception as e:
                    print(f"Failed to load {file}: {e}")
    total_hours = total_seconds / 3600
    return total_hours, file_count

# Ganti dengan path ke folder kamu
folder = "/home/nafis/code/text-to-speech/data/data/tito/wavs"
hours, count = get_total_audio_duration_and_count(folder)
print(f"Total audio duration: {hours:.2f} hours")
print(f"Total number of audio files: {count}")


Total audio duration: 4.05 hours
Total number of audio files: 3349


In [ ]:
CUDA_VISIBLE_DEVICES=2 nohup python convert_dac.py --path "/home/nafis/code/text-to-speech/data/data/tito/wavs/*.wav" > convert_dac.out & 

## dara

In [6]:
import csv
import json
import os

# path folder wav
wav_dir = "/home/nafis/code/text-to-speech/data/data/dara/wavs"
# input csv
csv_file = "/home/nafis/code/text-to-speech/data/data/dara/dara_transcript.txt"
# output json
json_file = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/dara.json"

data = []

with open(csv_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")  # karena pakai | di contohmu
    for row in reader:
        if len(row) < 2:
            continue  # skip baris yang ga valid
        filename = row[0].strip()
        text = row[1].strip()
        
        audio_path = os.path.join(wav_dir, filename)
        if not os.path.exists(audio_path):
            print(f"Warning: file {audio_path} tidak ditemukan.")
            continue

        # Ambil speaker ID dari nama file
        dara_filename = f"dara"

        data.append({
            "audio_path": audio_path,
            "text": text,
            "speaker": dara_filename
        })

# simpan jadi JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"JSON dataset tersimpan di {json_file}, total {len(data)} item")

JSON dataset tersimpan di /home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json/dara.json, total 4546 item


In [7]:
import torchaudio
import os

def get_total_audio_duration_and_count(folder_path, extensions={".wav", ".mp3", ".flac"}):
    total_seconds = 0.0
    file_count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                path = os.path.join(root, file)
                try:
                    waveform, sample_rate = torchaudio.load(path)
                    duration = waveform.shape[1] / sample_rate
                    total_seconds += duration
                    file_count += 1
                except Exception as e:
                    print(f"Failed to load {file}: {e}")
    total_hours = total_seconds / 3600
    return total_hours, file_count

# Ganti dengan path ke folder kamu
folder = "/home/nafis/code/text-to-speech/data/data/dara/wavs"
hours, count = get_total_audio_duration_and_count(folder)
print(f"Total audio duration: {hours:.2f} hours")
print(f"Total number of audio files: {count}")

Total audio duration: 5.39 hours
Total number of audio files: 4546


In [ ]:
CUDA_VISIBLE_DEVICES=2 nohup python convert_dac.py --path "/home/nafis/code/text-to-speech/data/data/dara/wavs/*.wav" > convert_dac.out & 